# Etapa 5 — Redes Bayesianas

Nesta etapa, vamos modelar as dependências entre as variáveis do domínio do Meio Ambiente usando uma Rede Bayesiana. O objetivo é criar um Grafo Acíclico Direcionado (DAG), estimar as Tabelas de Probabilidade Condicional (CPTs) a partir dos dados e realizar inferências.

**Bibliotecas necessárias:**
Usaremos o `pgmpy` para a modelagem e inferência, além do `pandas` para manipular os dados.


In [14]:
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

df_full = pd.read_csv("../dados/tabela_p4.csv", encoding="utf-8-sig")
df_original = df_full.copy()
variaveis_rede = ['CHUVA_FAIXA', 'TEMP_FAIXA', 'BIOMA', 'TIPO_ACAO', 'Multa_Alta']
df_treino = df_full[variaveis_rede].dropna(subset=variaveis_rede).astype(str)

##Precisamos definir de 4 a 6 variáveis e justificar as arestas (causa $\rightarrow$ efeito). Escolhemos 5 variáveis:
1. **CHUVA_FAIXA** : Condição climática (chuva).
2. **TEMP_FAIXA**: Condição climática (temperatura).
3. **BIOMA**: Região da ocorrência.
4. **TIPO_ACAO**: Tipo de infração/ação do IBAMA (ex: Rotina, Operação).
5. **Multa_Alta**: Nossa variável alvo (0 ou 1).

**Justificativa das Arestas:**
* Clima (`CHUVA_FAIXA` e `TEMP_FAIXA`) e o `BIOMA` influenciam diretamente as condições do terreno e as atividades na região, afetando o `TIPO_ACAO`.
* O `TIPO_ACAO` e o `BIOMA` influenciam diretamente a probabilidade de aplicação de uma `Multa_Alta`.

In [15]:
modelo = DiscreteBayesianNetwork([
    ('CHUVA_FAIXA', 'TIPO_ACAO'),
    ('TEMP_FAIXA', 'TIPO_ACAO'),
    ('BIOMA', 'TIPO_ACAO'),
    ('TIPO_ACAO', 'Multa_Alta'),
    ('BIOMA', 'Multa_Alta')  # <- Adicionar esta aresta
])

## Passo 2 — Estimar as CPTs a partir dos dados

Com o grafo desenhado, a rede precisa aprender as probabilidades. Como temos a tabela de dados completa, usamos a Estimativa de Máxima Verossimilhança (Maximum Likelihood Estimation - MLE). O `pgmpy` conta as frequências no DataFrame e constrói as tabelas.

In [16]:
# ==========================================
# Passo 2 — Estimar as CPTs com Suavização BDeu
# ==========================================
estimator = BayesianEstimator(modelo, df_treino)
cpts = estimator.get_parameters(prior_type='BDeu', equivalent_sample_size=10)

for cpd in cpts:
    modelo.add_cpds(cpd)

print("\nModelo é válido?", modelo.check_model())

cpt_multa = modelo.get_cpds('Multa_Alta')
print("\nTabela de Probabilidade Condicional para Multa_Alta:")
print(cpt_multa)


Modelo é válido? True

Tabela de Probabilidade Condicional para Multa_Alta:
+---------------+-----+---------------------+
| BIOMA         | ... | BIOMA(Pantanal)     |
+---------------+-----+---------------------+
| TIPO_ACAO     | ... | TIPO_ACAO(Rotina)   |
+---------------+-----+---------------------+
| Multa_Alta(0) | ... | 0.5323076923076923  |
+---------------+-----+---------------------+
| Multa_Alta(1) | ... | 0.46769230769230774 |
+---------------+-----+---------------------+


C:\Users\maria\AppData\Local\Temp\ipykernel_8616\1367048150.py:4: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(modelo, df_treino)


## Passo 3 — Inferência (Respondendo a consultas)

Vamos usar a Eliminação de Variáveis (`VariableElimination` do `pgmpy`) para responder a perguntas sobre o nosso modelo.

**Consulta:** Qual a probabilidade de a multa ser alta (`Multa_Alta`) sabendo que a ação do IBAMA foi uma "Operação"?

In [17]:
from pgmpy.inference import VariableElimination

# Criar o motor de inferência
inferencia = VariableElimination(modelo)

# Calcular a probabilidade
resultado = inferencia.query(variables=['Multa_Alta'], evidence={'TIPO_ACAO': 'Operação'})
print("Probabilidade de Multa Alta dado que a ação é uma 'Operação':")
print(resultado)

Probabilidade de Multa Alta dado que a ação é uma 'Operação':
+---------------+-------------------+
| Multa_Alta    |   phi(Multa_Alta) |
+===============+===================+
| Multa_Alta(0) |            0.4942 |
+---------------+-------------------+
| Multa_Alta(1) |            0.5058 |
+---------------+-------------------+


## Passo 4 — Independência Condicional (d-separação)

A d-separação permite verificar se duas variáveis são independentes dado o conhecimento de uma terceira. 

No nosso modelo, o `TIPO_ACAO` está no meio do caminho entre o clima (`CHUVA_FAIXA`) e a `Multa_Alta`. Vamos provar que se soubermos o `TIPO_ACAO`, a chuva deixa de influenciar diretamente a probabilidade da multa (o caminho fica bloqueado).

In [18]:
# ==========================================
# Passo 4 — Independência Condicional (d-separação e Colisor)
# ==========================================
print("\n--- Teste de d-separação (Colisor) ---")

# 1. Sem evidência
caminho_aberto = modelo.is_dconnected('CHUVA_FAIXA', 'Multa_Alta')
print(f"Caminho ativo entre Chuva e Multa (sem evidência): {caminho_aberto}")

# 2. Com evidência no nó TIPO_ACAO (atuando como colisor)
caminho_bloqueado = modelo.is_dconnected('CHUVA_FAIXA', 'Multa_Alta', observed=['TIPO_ACAO'])
print(f"Caminho ativo entre Chuva e Multa (sabendo o TIPO_ACAO): {caminho_bloqueado}")


--- Teste de d-separação (Colisor) ---
Caminho ativo entre Chuva e Multa (sem evidência): True
Caminho ativo entre Chuva e Multa (sabendo o TIPO_ACAO): True


## Passo 5 — Economia de Parâmetros

Uma das grandes vantagens das Redes Bayesianas é a redução do número de parâmetros (números) necessários em comparação com uma tabela de distribuição conjunta completa.

Vamos calcular quantos parâmetros a nossa rede economizou.

In [19]:
import numpy as np

# Calcular parâmetros da distribuição conjunta completa
categorias = {var: modelo.get_cardinality(var) for var in modelo.nodes()}
total_conjunta = np.prod(list(categorias.values())) - 1

# Calcular parâmetros da Rede Bayesiana
total_rede = sum([len(cpt.values.flatten()) - (len(cpt.values.flatten()) // cpt.variable_card) for cpt in modelo.get_cpds()])

print(f"Parâmetros necessários na Conjunta Completa: {total_conjunta}")
print(f"Parâmetros necessários na Rede Bayesiana: {total_rede}")
print(f"Economia total: a nossa rede poupa {total_conjunta - total_rede} parâmetros!")

Parâmetros necessários na Conjunta Completa: 671
Parâmetros necessários na Rede Bayesiana: 257
Economia total: a nossa rede poupa 414 parâmetros!


In [20]:
# ==========================================
# Passo 6 — Exportação das Probabilidades
# ==========================================
p_multa_alta_lista = []

for _, row in df_original.iterrows():
    evidencia = {}
    for var in ['CHUVA_FAIXA', 'TEMP_FAIXA', 'BIOMA', 'TIPO_ACAO']:
        if pd.notna(row[var]):
            evidencia[var] = str(row[var])
            
    try:
        res = inferencia.query(
            variables=['Multa_Alta'],
            evidence=evidencia if len(evidencia) > 0 else None,
            show_progress=False
        )
        idx_1 = list(res.state_names['Multa_Alta']).index('1') if '1' in res.state_names['Multa_Alta'] else 1
        p_val = float(res.values[idx_1])
    except Exception:
        p_val = np.nan
        
    p_multa_alta_lista.append(p_val)

df_probs = pd.DataFrame({'p_multa_alta': p_multa_alta_lista})
df_probs.to_csv('../dados/probs_rede.csv', index=False, encoding='utf-8-sig')

print(f"\nFicheiro 'probs_rede.csv' exportado com sucesso! Total de linhas: {len(df_probs)}")


Ficheiro 'probs_rede.csv' exportado com sucesso! Total de linhas: 2639
